In [5]:
import os
from pathlib import Path
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import scipy.stats
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import shap
from sklearn.model_selection import train_test_split, ParameterGrid
from scipy.signal import butter, sosfilt

In [21]:
SAMPLE_RATE = 16_000

DCASE2024_ROOT_PATH = Path.cwd().parent / "data/raw/dcase-2024"

DCASE2024_TRAIN_PATH = DCASE2024_ROOT_PATH / "Train"
DCASE2024_DEV_PATH = DCASE2024_ROOT_PATH / "Dev"
DCASE2024_EVAL_PATH = DCASE2024_ROOT_PATH / "Eval"

DCASE2022_ROOT_PATH = Path.cwd().parent / "data/raw/dcase-2022"

DCASE2022_TRAIN_PATH = DCASE2022_ROOT_PATH / "Train"
DCASE2022_DEV_PATH = DCASE2022_ROOT_PATH / "Dev"

MACHINE_TYPES = ['fan']
SECTIONS = ['section_00']


In [ ]:
def load_audio(audio):
    signal, sample_rate = librosa.load(audio, mono=True, sr=SAMPLE_RATE)

    return signal, sample_rate

def pad_or_trim(signal, target_len):
    if len(signal) > target_len:
        return signal[:target_len]
    elif len(signal) < target_len:
        return np.pad(signal, (0, target_len - len(signal)), 'constant')
   
    return signal

In [53]:
def read_dcase_dev_set(
    path,  
    only_machine_type=[], 
    only_section=[]):

    signals_train, y_train = [], []
    signals_test, y_test = [], []

    for machine_dir in path.iterdir():
        machine_type = machine_dir.name

        if only_machine_type and machine_type not in only_machine_type: continue

        for audio_set in ['train', 'test']:
            audios_dir = machine_dir / audio_set
            wavs = list(audios_dir.glob("*.wav"))

            for audio_file in tqdm(wavs, desc=f"{machine_type} - {audio_set}"):
                split_name = audio_file.name.split('_')
                section = f'{split_name[0]}_{split_name[1]}'

                if only_section and section not in only_section: continue
                
                try:
                    signal, _ = load_audio(audio_file)

                    if audio_set == "train":
                        label = 0
                        signals_train.append(signal)
                        y_train.append(label)

                    else:
                        label = 1 if "anomaly" in audio_file.name.lower() else 0
                        signals_test.append(signal)
                        y_test.append(label)

                except Exception as e:
                    print(f"Erro ao processar {audio_file.name}: {e}")
    
    return np.array(signals_train), \
           np.array(signals_test), \
           np.array(y_train), \
           np.array(y_test)

(signals_train, 
signals_test, 
labels_train, 
labels_test, 
) = read_dcase_dev_set(DCASE2022_DEV_PATH, only_machine_type=MACHINE_TYPES, only_section=SECTIONS)

fan - test: 100%|██████████| 600/600 [00:00<00:00, 5017.68it/s]


In [54]:
def preprocess_data(signals, 
                    padding_or_trim=True, 
                    denoise_method=None, 
                    normalize_method=None):

    new_signals = []
    target_len = 10 * SAMPLE_RATE

    for signal in signals:
        if padding_or_trim:
            signal = pad_or_trim(signal, target_len)
        if denoise_method:
            signal = denoise_method(signal)
        if normalize_method:
            signal = normalize_method(signal)
        
        new_signals.append(signal)

    return np.array(new_signals)

signals_train = preprocess_data(signals_train)
signals_test = preprocess_data(signals_test)

In [67]:
def create_windows(signal, size, stride):
    frame_length = int(size*SAMPLE_RATE)
    hop_length = int(stride*SAMPLE_RATE)

    windows = librosa.util.frame(signal, frame_length=frame_length, hop_length=hop_length)

    return windows.T


windows = create_windows(signals_train[0], 0.050, 0.050)

In [73]:
def get_mfcc(signal):
    mfcc = librosa.feature.mfcc(y=signal, sr=SAMPLE_RATE, n_mfcc=128)

    return mfcc.astype(np.float32)

get_mfcc(windows[0]).shape

c:\Users\josel\OneDrive\Desktop\NCIA\.venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=800
  warnings.warn(


(128, 2)